# 02 — Nettoyage, segmentation et analyse

**Entree** : `data/produits_bruts.csv` (produit par `01_collecte.ipynb` ou `src/collecte.py`)
**Sortie** : `data/produits_clean.csv` et `data/kpi_verifies.csv`

La segmentation se fait **par quantiles a l'interieur de chaque categorie** : un produit
est compare aux autres de sa categorie, pas au marche entier. Comparer une echarpe a une
montre n'aurait aucun sens.


In [1]:
# --- D1 : PROFILING — état des lieux de la qualité ---
print("=== DIAGNOSTIC QUALITÉ ===\n")
print("Dimensions du tableau :", df.shape)          # (lignes, colonnes)
print("\nValeurs manquantes par colonne :")
print(df.isnull().sum())                            # combien de NaN par colonne
print("\nDoublons (même id) :", df.duplicated(subset=["id"]).sum())
print("\nDevises présentes :", df["devise"].unique())

=== DIAGNOSTIC QUALITÉ ===

Dimensions du tableau : (119, 9)

Valeurs manquantes par colonne :
id             0
titre          0
marque         0
categorie      0
prix           0
prix_barre    64
devise         0
retailer       0
requete        0
dtype: int64

Doublons (même id) : 0

Devises présentes : <StringArray>
['USD']
Length: 1, dtype: str


In [2]:
# --- D2 : NETTOYAGE ---
# On part d'une copie propre
df_clean = df.copy()

# 1. Enlever les produits sans prix (inexploitables pour une analyse de prix)
df_clean = df_clean[df_clean["prix"].notna()]

# 2. Enlever les doublons éventuels (même id)
df_clean = df_clean.drop_duplicates(subset=["id"])

# 3. Ne garder que les prix en USD (pour comparer ce qui est comparable)
df_clean = df_clean[df_clean["devise"] == "USD"]

# 4. Créer une colonne "remise" quand il y a un prix barré
df_clean["remise_pct"] = (
    (df_clean["prix_barre"] - df_clean["prix"]) / df_clean["prix_barre"] * 100
).round(1)

print("Avant nettoyage :", len(df), "produits")
print("Après nettoyage :", len(df_clean), "produits")
df_clean.head()

Avant nettoyage : 119 produits
Après nettoyage : 119 produits


,id,titre,marque,categorie,prix,prix_barre,devise,retailer,requete,remise_pct
0,rP7Lp38,Leather Handbag,Bottega Veneta,Handbags,1745.0,2425.0,USD,jomashop.com,leather handbag,28.0
1,gW1gxYV,Prada Darling Leather Handbag,Prada,"Handbags, Wallets & Cases",3550.0,NaN,USD,prada.com,leather handbag,NaN
2,t2JZlpl,Amazona 23 Leather Handbag,LOEWE,Handbags,2774.0,4170.0,USD,luosophy.com,leather handbag,33.5
3,oqE5xBI,Vita Leather Handbag,Golden Goose,Handbags,724.0,1278.0,USD,us.thahab.com,leather handbag,43.3
4,JAOAQX5,Darling leather handbag,Prada,"Handbags, Wallets & Cases",3550.0,NaN,USD,prada.com,leather handbag,NaN


In [3]:
# --- E1 : Positionnement prix par marque ---
analyse_marque = df_clean.groupby("marque").agg(
    nb_produits=("id", "count"),
    prix_moyen=("prix", "mean"),
    prix_min=("prix", "min"),
    prix_max=("prix", "max")
).round(0).sort_values("prix_moyen", ascending=False)

print("=== POSITIONNEMENT PRIX PAR MARQUE ===")
analyse_marque

=== POSITIONNEMENT PRIX PAR MARQUE ===


,nb_produits,prix_moyen,prix_min,prix_max
marque,,,,
Louis Vuitton,2,38210.0,421.0,76000.0
Chopard,2,26398.0,23840.0,28955.0
Audemars Piguet,2,23775.0,20155.0,27395.0
Rolex,2,16730.0,12950.0,20511.0
Jaeger-LeCoultre,1,10500.0,10500.0,10500.0
...,...,...,...,...
Fendi,1,50.0,50.0,50.0
Le Specs,1,49.0,49.0,49.0
Dior,1,49.0,49.0,49.0


In [4]:
# --- E2 : Positionnement par catégorie de recherche ---
analyse_categorie = df_clean.groupby("requete").agg(
    nb_produits=("id", "count"),
    prix_moyen=("prix", "mean"),
    prix_median=("prix", "median"),
    prix_min=("prix", "min"),
    prix_max=("prix", "max")
).round(0).sort_values("prix_moyen", ascending=False)

print("=== ÉCARTS DE PRIX PAR CATÉGORIE ===")
analyse_categorie

=== ÉCARTS DE PRIX PAR CATÉGORIE ===


,nb_produits,prix_moyen,prix_median,prix_min,prix_max
requete,,,,,
luxury watch,20,12372.0,4875.0,230.0,76000.0
leather handbag,20,1821.0,845.0,190.0,5040.0
wool coat,20,1241.0,619.0,45.0,6330.0
leather shoes,19,319.0,100.0,14.0,1667.0
sunglasses,20,242.0,205.0,49.0,619.0
silk scarf,20,232.0,187.0,45.0,665.0


In [ ]:
# Segmentation par quantiles, À L'INTÉRIEUR de chaque catégorie
def segmenter(groupe):
    q33 = groupe["prix"].quantile(0.33)
    q66 = groupe["prix"].quantile(0.66)
    def label(prix):
        if prix <= q33:
            return "1. Accessible"
        elif prix <= q66:
            return "2. Mid"
        else:
            return "3. Luxe"
    groupe["segment"] = groupe["prix"].apply(label)
    return groupe

df_clean = df_clean.groupby("requete", group_keys=False).apply(segmenter)

In [ ]:
stats_segment = df_clean.groupby("segment").agg(
    nb_produits=("id", "count"),
    prix_moyen=("prix", "mean"),
    prix_median=("prix", "median"),      # LA vérité sur une distribution asymétrique
    prix_ecart_type=("prix", "std")       # la dispersion
).round(0)

In [5]:
remise_par_segment = df_clean.groupby("segment").apply(
    lambda g: (g["prix_barre"] - g["prix"]).sum() / g["prix_barre"].sum()
)
print(remise_par_segment)

segment
1. Accessible    0.423345
2. Mid           0.365028
3. Luxe          0.198861
dtype: float64


In [6]:
# Sauvegarder le tableau nettoyé dans data/ (pour Power BI ensuite)
df_clean.to_csv("../data/produits_clean.csv", index=False)
print("Fichier sauvegardé : data/produits_clean.csv")
print("Prêt pour Power BI !")

Fichier sauvegardé : data/produits_clean.csv
Prêt pour Power BI !
